# Lab 13 - Univariate Time Series Analysis: PM2.5 Forecasting

This notebook applies **Lab 13 - Univariate Time Series Analysis** to hourly PM2.5 values for one city.


## Lab 13 concepts used

- Build a single-variable city time series.
- Resample hourly observations and handle gaps.
- Plot rolling averages and seasonal patterns.
- Create lag features.
- Compare a persistence baseline with machine-learning lag models.

This notebook forecasts `PM2_5_ug_m3` from its own previous values only, so it is separate from the multivariate regression labs.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'AML Assignment' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT


In [ ]:
DATASET_FILENAME = 'global_urban_smog_pm25_hourly.csv'
matches = sorted((PROJECT_ROOT / 'Datasets').glob(f'*/{DATASET_FILENAME}'))
if not matches:
    raise FileNotFoundError(f'Could not find {DATASET_FILENAME} under {PROJECT_ROOT / "Datasets"}')
DATASET_PATH = matches[0]
data = pd.read_csv(DATASET_PATH)
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values(['Timestamp', 'City']).reset_index(drop=True)
print(DATASET_PATH)
data.head()


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error


In [ ]:
city_counts = data['City'].value_counts()
selected_city = city_counts.index[0]
selected_city, city_counts.iloc[0]


In [ ]:
city_series = (
    data[data['City'] == selected_city]
    .set_index('Timestamp')
    .sort_index()['PM2_5_ug_m3']
    .resample('h')
    .mean()
    .interpolate(method='time')
)
city_series = city_series.rename('PM2_5_ug_m3')
city_series.head(), city_series.tail(), city_series.shape


In [ ]:
plt.figure(figsize=(12, 4))
city_series.plot(alpha=0.55, label='Hourly PM2.5')
city_series.rolling(24).mean().plot(label='24-hour rolling mean')
city_series.rolling(24 * 7).mean().plot(label='7-day rolling mean')
plt.title(f'{selected_city}: hourly PM2.5 with rolling means')
plt.ylabel('PM2.5')
plt.legend()
plt.show()


In [ ]:
seasonality_df = city_series.to_frame()
seasonality_df['hour'] = seasonality_df.index.hour
seasonality_df['dayofweek'] = seasonality_df.index.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=seasonality_df, x='hour', y='PM2_5_ug_m3', estimator='mean', errorbar=None, ax=axes[0])
axes[0].set_title('Average PM2.5 by hour')
sns.lineplot(data=seasonality_df, x='dayofweek', y='PM2_5_ug_m3', estimator='mean', errorbar=None, ax=axes[1])
axes[1].set_title('Average PM2.5 by day of week')
plt.tight_layout()
plt.show()


In [ ]:
def make_lag_frame(series, lags=(1, 2, 3, 6, 12, 24, 48, 168)):
    frame = series.to_frame()
    for lag in lags:
        frame[f'lag_{lag}'] = frame[series.name].shift(lag)
    frame['rolling_24_mean'] = frame[series.name].shift(1).rolling(24).mean()
    frame['rolling_168_mean'] = frame[series.name].shift(1).rolling(168).mean()
    frame['hour'] = frame.index.hour
    frame['dayofweek'] = frame.index.dayofweek
    return frame.dropna()

lag_df = make_lag_frame(city_series)
lag_df.head()


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

split_idx = int(len(lag_df) * 0.8)
train_df = lag_df.iloc[:split_idx]
test_df = lag_df.iloc[split_idx:]
feature_cols = [col for col in lag_df.columns if col != 'PM2_5_ug_m3']
X_train = train_df[feature_cols]
y_train = train_df['PM2_5_ug_m3']
X_test = test_df[feature_cols]
y_test = test_df['PM2_5_ug_m3']

baseline_pred = X_test['lag_1']
ridge_model = Pipeline(steps=[('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])
rf_model = RandomForestRegressor(n_estimators=120, max_depth=12, min_samples_leaf=10, n_jobs=-1, random_state=42)

ridge_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

forecast_results = pd.DataFrame([
    {'model': 'Persistence baseline', 'MAE': mean_absolute_error(y_test, baseline_pred), 'RMSE': root_mean_squared_error(y_test, baseline_pred), 'R2': r2_score(y_test, baseline_pred)},
    {'model': 'Ridge lag model', 'MAE': mean_absolute_error(y_test, ridge_pred), 'RMSE': root_mean_squared_error(y_test, ridge_pred), 'R2': r2_score(y_test, ridge_pred)},
    {'model': 'Random forest lag model', 'MAE': mean_absolute_error(y_test, rf_pred), 'RMSE': root_mean_squared_error(y_test, rf_pred), 'R2': r2_score(y_test, rf_pred)},
]).sort_values('RMSE')
forecast_results


In [ ]:
forecast_plot = pd.DataFrame({
    'actual': y_test,
    'baseline': baseline_pred,
    'ridge': ridge_pred,
    'random_forest': rf_pred,
}).tail(24 * 7)

plt.figure(figsize=(12, 4))
plt.plot(forecast_plot.index, forecast_plot['actual'], label='Actual', linewidth=2)
plt.plot(forecast_plot.index, forecast_plot['baseline'], label='Persistence', alpha=0.75)
plt.plot(forecast_plot.index, forecast_plot['ridge'], label='Ridge', alpha=0.75)
plt.plot(forecast_plot.index, forecast_plot['random_forest'], label='Random forest', alpha=0.75)
plt.title(f'{selected_city}: last test week PM2.5 forecast')
plt.ylabel('PM2.5')
plt.legend()
plt.show()


## What was learned from Lab 13

A univariate time-series framing is useful for PM2.5 forecasting and for explaining temporal patterns. The persistence baseline is essential: a more complex lag model should only be preferred if it improves meaningfully on this simple benchmark.
